# Fraud Detection Model Training - XGBoost v2 (Robust)

This notebook is a cell-by-cell walkthrough of the production training script
`xgboost_model_v2_robust.py`. It is the SAME code, broken into stages so each
step's output can be inspected individually, rather than a separate/simplified
version - running this notebook top to bottom reproduces exactly the same
saved model and metrics as running the script.

**Pipeline stages covered:** load data -> engineer contextual features ->
split -> SMOTENC resampling -> inject missingness -> one-hot encode -> train
XGBoost -> evaluate (clean + noise-corrupted) -> per-fraud-type recall -> save.


## 1. Imports

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, balanced_accuracy_score, average_precision_score, classification_report
from imblearn.over_sampling import SMOTENC
import xgboost as xgb
import warnings
import os
import gc

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
MISSING_FRAC = 0.08  # 8% missingness injected into training cost columns

DATA_PATH = os.path.expanduser(
    "~/synthea_kenya/output/kenyanised/master_claims_sample_500k_13pct_v2.csv"
)
OUTPUT_DIR = os.path.expanduser("~/synthea_kenya/output/model")
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Load the dataset

Reads `master_claims_sample_500k_13pct_v2.csv` - 500,000 claims, ~13% fraud,
including collusion_ring examples (the earlier, now-retired version of this
file had none). Categorical columns are kept as raw strings here because
SMOTENC (later) needs raw categories, not one-hot columns.

In [16]:
NEEDED_COLS = list(set([
    "BASE_COST_KES", "CLAIM_AMOUNT_KES", "PAYER_COVERAGE_KES", "CLAIM_TO_BASE_RATIO",
    "AGE", "LENGTH_OF_STAY_HOURS", "ENCOUNTER_TYPE", "GENDER", "PATIENT_COUNTY",
    "IS_FRAUD", "PRIMARY_CONDITION", "PROVIDER_ID", "SERVICE_DATE", "ENCOUNTER_ID",
    "FRAUD_TYPE",
]))
df = pd.read_csv(DATA_PATH, usecols=NEEDED_COLS)
print(f"Dataset shape: {df.shape}, fraud rate: {df['IS_FRAUD'].mean()*100:.2f}%")
df["FRAUD_TYPE"].value_counts()


Dataset shape: (500000, 15), fraud rate: 13.00%


FRAUD_TYPE
none               435000
upcoding            27083
collusion_ring      21667
phantom_billing     16250
Name: count, dtype: int64

## 3. Fix the one-hot category vocabulary

Categories are listed from the **whole** dataset up front (not just train) so
every one-hot column exists consistently across train/val/test regardless of
which split a rare category lands in. This never looks at the fraud label, so
it isn't a leakage risk - it's the same thing a fitted LabelEncoder/
pd.get_dummies over the full dataframe would have done before.

In [17]:
categorical_features = ["ENCOUNTER_TYPE", "GENDER", "PATIENT_COUNTY"]
for col in categorical_features:
    df[col] = df[col].astype(str)

onehot_categories = {col: sorted(df[col].unique().tolist()) for col in categorical_features}
encoded_categoricals = [f"{col}_{cat}" for col in categorical_features for cat in onehot_categories[col]]
for col, cats in onehot_categories.items():
    print(f"{col}: {len(cats)} categories")


def onehot_encode(df_slice: pd.DataFrame, categorical_cols, categories: dict) -> pd.DataFrame:
    """Deterministic one-hot expansion using a FIXED category list (not
    pd.get_dummies' auto-discovered columns), so train/val/test and the live
    dashboard all produce identical column sets regardless of which
    categories happen to appear in a given slice."""
    out = df_slice.drop(columns=categorical_cols).copy()
    for col in categorical_cols:
        values = df_slice[col].astype(str)
        for cat in categories[col]:
            out[f"{col}_{cat}"] = (values == cat).astype(int)
    return out


ENCOUNTER_TYPE: 10 categories
GENDER: 2 categories
PATIENT_COUNTY: 20 categories


## 4. Train/validation/test split

70/15/15 split, stratified on the fraud label so each split keeps the same
~13% fraud rate. Same `random_state=42` as every other script in this
project, so row membership is identical and results are comparable run to
run.

In [18]:
y_full = df["IS_FRAUD"]
idx_train, idx_temp = train_test_split(
    df.index, test_size=0.30, random_state=RANDOM_STATE, stratify=y_full
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_full.loc[idx_temp]
)
print(f"Train/Val/Test sizes: {len(idx_train)}/{len(idx_val)}/{len(idx_test)}")

# Captured now, before anything else touches df - needed later for the
# per-fraud-type recall report.
test_fraud_type = df.loc[idx_test, "FRAUD_TYPE"].copy() if "FRAUD_TYPE" in df.columns else None


Train/Val/Test sizes: 350000/75000/75000


## 5. Engineer contextual features (fit on TRAIN split only)

Three derived features, each a group-based aggregation statistic computed
from the training split only (then looked up/applied across train/val/test -
no peeking at validation or test data when building these):

- **COST_RATIO** - claim amount divided by the median claim amount for the
  same (condition, county) pair.
- **PROVIDER_Z_SCORE** - how many standard deviations a claim is from that
  provider's own average claim amount.
- **PROVIDER_DAILY_VOLUME** - how many claims that provider filed on that
  same calendar day (uses the full dataset since it only counts co-occurring
  rows and never touches the label - no leakage).

In [19]:
train_df = df.loc[idx_train]

# --- Regional cost anomaly ratio ---
group_cols = ["PRIMARY_CONDITION", "PATIENT_COUNTY"]
group_median = train_df.groupby(group_cols)["CLAIM_AMOUNT_KES"].median()
global_median = train_df["CLAIM_AMOUNT_KES"].median()

median_lookup = df[group_cols].apply(tuple, axis=1).map(group_median)
median_lookup = median_lookup.fillna(global_median)
df["COST_RATIO"] = (df["CLAIM_AMOUNT_KES"] / median_lookup.replace(0, np.nan)).fillna(1.0)

# --- Provider peer deviation (z-score) ---
provider_mean = train_df.groupby("PROVIDER_ID")["CLAIM_AMOUNT_KES"].mean()
provider_std = train_df.groupby("PROVIDER_ID")["CLAIM_AMOUNT_KES"].std()
global_mean = train_df["CLAIM_AMOUNT_KES"].mean()
global_std = train_df["CLAIM_AMOUNT_KES"].std()

mean_lookup = df["PROVIDER_ID"].map(provider_mean).fillna(global_mean)
std_lookup = df["PROVIDER_ID"].map(provider_std).fillna(global_std).replace(0, global_std)
df["PROVIDER_Z_SCORE"] = ((df["CLAIM_AMOUNT_KES"] - mean_lookup) / std_lookup).fillna(0.0)

# --- Time-based volatility (same-day claim count per provider) ---
daily_counts = df.groupby(["PROVIDER_ID", "SERVICE_DATE"])["ENCOUNTER_ID"].transform("count")
df["PROVIDER_DAILY_VOLUME"] = daily_counts

print("New features added: COST_RATIO, PROVIDER_Z_SCORE, PROVIDER_DAILY_VOLUME")
df[["COST_RATIO", "PROVIDER_Z_SCORE", "PROVIDER_DAILY_VOLUME"]].describe()


New features added: COST_RATIO, PROVIDER_Z_SCORE, PROVIDER_DAILY_VOLUME


,COST_RATIO,PROVIDER_Z_SCORE,PROVIDER_DAILY_VOLUME
count,500000.000000,500000.000000,500000.000000
mean,1.774691,0.004382,1.441340
std,2.932962,1.082764,0.835701
min,0.023238,-39.543425,1.000000
25%,0.621707,-0.352366,1.000000
50%,1.000000,-0.221873,1.000000
75%,1.604149,-0.036328,2.000000
max,75.822952,78.789057,11.000000


## 6. Assemble the feature set

In [20]:
COST_FEATURES = ["BASE_COST_KES", "CLAIM_AMOUNT_KES", "PAYER_COVERAGE_KES"]
RATIO_FEATURE = ["CLAIM_TO_BASE_RATIO"]
CONTEXT_FEATURES = ["COST_RATIO", "PROVIDER_Z_SCORE", "PROVIDER_DAILY_VOLUME"]
OTHER_NUMERICAL = ["AGE", "LENGTH_OF_STAY_HOURS"]  # PATIENT_INCOME excluded - not applicable to unemployed/informal-sector patients

RAW_FEATURE_COLUMNS = COST_FEATURES + RATIO_FEATURE + CONTEXT_FEATURES + OTHER_NUMERICAL + categorical_features
FEATURE_COLUMNS = COST_FEATURES + RATIO_FEATURE + CONTEXT_FEATURES + OTHER_NUMERICAL + encoded_categoricals
NOISY_COLUMNS = COST_FEATURES + RATIO_FEATURE + ["COST_RATIO", "PROVIDER_Z_SCORE"]

print(f"Raw features going into SMOTENC ({len(RAW_FEATURE_COLUMNS)}): {RAW_FEATURE_COLUMNS}")
print(f"Final (one-hot) features for the model ({len(FEATURE_COLUMNS)} total columns)")

X_train_raw = df.loc[idx_train, RAW_FEATURE_COLUMNS]
X_val_raw = df.loc[idx_val, RAW_FEATURE_COLUMNS]
X_test_raw = df.loc[idx_test, RAW_FEATURE_COLUMNS]
y_train = y_full.loc[idx_train]
y_val = y_full.loc[idx_val]
y_test = y_full.loc[idx_test]

del train_df, median_lookup, mean_lookup, std_lookup, daily_counts
gc.collect()


Raw features going into SMOTENC (12): ['BASE_COST_KES', 'CLAIM_AMOUNT_KES', 'PAYER_COVERAGE_KES', 'CLAIM_TO_BASE_RATIO', 'COST_RATIO', 'PROVIDER_Z_SCORE', 'PROVIDER_DAILY_VOLUME', 'AGE', 'LENGTH_OF_STAY_HOURS', 'ENCOUNTER_TYPE', 'GENDER', 'PATIENT_COUNTY']
Final (one-hot) features for the model (41 total columns)


0

## 7. SMOTENC oversampling (handles class imbalance)

Fraud is only ~13% of claims. SMOTENC synthetically generates more fraud
examples for the TRAINING set only (never val/test). It's the categorical-
aware version of SMOTE: it operates on the raw category labels directly
instead of one-hot columns, which matters a lot here - plain SMOTE/SMOTEENN
on the one-hot-expanded (41-dimension) feature space did not finish in
several minutes during testing, while SMOTENC finished the full training
split in under a minute.

In [21]:
cat_idx = [RAW_FEATURE_COLUMNS.index(c) for c in categorical_features]
smote_nc = SMOTENC(categorical_features=cat_idx, random_state=RANDOM_STATE)
X_train_res_raw, y_train_res = smote_nc.fit_resample(X_train_raw, y_train)
print(f"Before: {y_train.value_counts().to_dict()}  After: {pd.Series(y_train_res).value_counts().to_dict()}")


Before: {0: 304500, 1: 45500}  After: {0: 304500, 1: 304500}


## 8. Inject missingness into the resampled training set

8% of values in every cost-related column are randomly set to NaN - AFTER
resampling, since SMOTE's nearest-neighbour interpolation can't operate on
NaNs. This teaches XGBoost's native missing-value handling to cope with
incomplete claims, which is realistic for messy real-world data feeds and is
what makes this model "robust" rather than brittle to anything less than
perfectly clean input.

In [22]:
rng = np.random.default_rng(RANDOM_STATE)
X_train_res_raw = X_train_res_raw.reset_index(drop=True)
for col in NOISY_COLUMNS:
    mask = rng.random(len(X_train_res_raw)) < MISSING_FRAC
    X_train_res_raw.loc[mask, col] = np.nan

missing_pct_check = X_train_res_raw[NOISY_COLUMNS].isna().mean() * 100
print("Actual missing % achieved per column:")
missing_pct_check


Actual missing % achieved per column:


BASE_COST_KES          7.975205
CLAIM_AMOUNT_KES       7.955172
PAYER_COVERAGE_KES     7.983908
CLAIM_TO_BASE_RATIO    7.978161
COST_RATIO             8.032512
PROVIDER_Z_SCORE       8.030213
dtype: float64

## 9. One-hot encode (now, after resampling)

One-hot encoding happens here rather than before resampling - it's cheap and
deterministic with the fixed category list from step 3, and doing it after
resampling is what avoids the SMOTE-on-high-dimensional-one-hot slowdown
described in step 7.

In [23]:
X_train_res = onehot_encode(X_train_res_raw, categorical_features, onehot_categories)[FEATURE_COLUMNS]
X_val = onehot_encode(X_val_raw, categorical_features, onehot_categories)[FEATURE_COLUMNS]
X_test = onehot_encode(X_test_raw, categorical_features, onehot_categories)[FEATURE_COLUMNS]
print(f"Final training matrix shape: {X_train_res.shape}")


Final training matrix shape: (609000, 41)


## 10. Train XGBoost

Regularized hyperparameters (shallower trees, lower learning rate, more
estimators with early stopping) relative to the original fragile v1 model,
to reduce overfitting to exact clean values. `scale_pos_weight` is computed
dynamically from the actual post-resampling class balance rather than a
fixed number, since resampling changes that ratio.

In [24]:
neg = sum(y_train_res == 0)
pos = sum(y_train_res == 1)
scale_pos_weight = neg / pos
print(f"Dynamic scale_pos_weight: {scale_pos_weight:.3f}")

model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.7,
    learning_rate=0.05,
    n_estimators=1000,
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
    n_jobs=2, tree_method="hist",
)
model.fit(X_train_res, y_train_res, eval_set=[(X_val, y_val)], verbose=False)
print(f"Best iteration: {model.best_iteration} (of 1000 max)")


Dynamic scale_pos_weight: 1.000
Best iteration: 998 (of 1000 max)


## 11. Evaluate

Evaluated on the clean test set, plus the same 5%/10% synthetic
noise+missingness corruption used in `robustness_test.py`, so results are
directly comparable to the original (fragile) v1 model.

In [25]:
def evaluate(X, y, label):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    f1 = f1_score(y, y_pred)
    bal_acc = balanced_accuracy_score(y, y_pred)
    pr_auc = average_precision_score(y, y_proba)
    print(f"\n{label}")
    print(f"F1: {f1:.4f}  Balanced Accuracy: {bal_acc:.4f}  PR-AUC: {pr_auc:.4f}")
    return {"experiment": label, "f1": f1, "balanced_accuracy": bal_acc, "pr_auc": pr_auc}


def corrupt(X, noise_frac, seed):
    """Same corruption function as robustness_test.py: adds Gaussian noise
    scaled to each column's std, then randomly masks values to NaN, at the
    given fraction - simulating a messy real-world claims feed."""
    r = np.random.default_rng(seed)
    Xc = X.copy()
    numeric_cols = [c for c in FEATURE_COLUMNS if c not in encoded_categoricals]
    for col in numeric_cols:
        std = Xc[col].std()
        noise = r.normal(loc=0, scale=std * noise_frac, size=len(Xc))
        Xc[col] = Xc[col] + noise
    mask = r.random(Xc.shape) < noise_frac
    Xc = Xc.mask(mask)
    return Xc


results = []
results.append(evaluate(X_test, y_test, "v2 - CLEAN test set"))
for noise_frac in [0.05, 0.10]:
    X_noisy = corrupt(X_test, noise_frac, seed=RANDOM_STATE)
    results.append(evaluate(X_noisy, y_test, f"v2 - {int(noise_frac*100)}% noise+missingness test set"))



v2 - CLEAN test set
F1: 0.7930  Balanced Accuracy: 0.9072  PR-AUC: 0.9059

v2 - 5% noise+missingness test set
F1: 0.5328  Balanced Accuracy: 0.8134  PR-AUC: 0.7685

v2 - 10% noise+missingness test set
F1: 0.4356  Balanced Accuracy: 0.7548  PR-AUC: 0.6693


In [26]:
print(classification_report(y_test, model.predict(X_test), target_names=["Legitimate", "Fraud"]))


              precision    recall  f1-score   support

  Legitimate       0.98      0.95      0.97     65250
       Fraud       0.74      0.86      0.79      9750

    accuracy                           0.94     75000
   macro avg       0.86      0.91      0.88     75000
weighted avg       0.95      0.94      0.94     75000



## 12. Per-fraud-type recall

The number that matters most for the collusion_ring / GraphSAGE story -
computed directly from THIS saved model (not a separate throwaway model), so
it's guaranteed consistent with what the model actually does in production.

In [27]:
fraud_type_rows = []
if test_fraud_type is not None:
    y_proba_test = model.predict_proba(X_test)[:, 1]
    y_pred_test = model.predict(X_test)
    for ft in sorted(test_fraud_type.unique()):
        if ft == "none":
            continue
        ft_mask = (test_fraud_type == ft).values
        n = int(ft_mask.sum())
        caught = int(y_pred_test[ft_mask].sum())
        recall = caught / n if n else float("nan")
        mean_score = float(y_proba_test[ft_mask].mean()) if n else float("nan")
        print(f"  {ft:<18} n={n:<6} caught={caught:<6} recall={recall:.4f} mean_score={mean_score:.4f}")
        fraud_type_rows.append({"fraud_type": ft, "n": n, "caught": caught, "recall": recall, "mean_score": mean_score})
pd.DataFrame(fraud_type_rows)


  collusion_ring     n=3246   caught=2239   recall=0.6898 mean_score=0.6257
  phantom_billing    n=2410   caught=2381   recall=0.9880 mean_score=0.9875
  upcoding           n=4094   caught=3772   recall=0.9213 mean_score=0.9201


,fraud_type,n,caught,recall,mean_score
0,collusion_ring,3246,2239,0.689772,0.625715
1,phantom_billing,2410,2381,0.987967,0.987477
2,upcoding,4094,3772,0.921348,0.920090


## 13. Save the model and results

Saves to the same filenames the production scripts (`export_model_artifacts.py`,
`app.py`, the dashboard) all read from - running this notebook end to end
updates the live, deployed model, not a separate copy.

In [28]:
model.save_model(f"{OUTPUT_DIR}/xgboost_fraud_model_v2_robust.json")
pd.DataFrame(results).to_csv(f"{OUTPUT_DIR}/v2_robust_results.csv", index=False)
if fraud_type_rows:
    pd.DataFrame(fraud_type_rows).to_csv(f"{OUTPUT_DIR}/per_fraudtype_recall_v2data.csv", index=False)
print(f"Model saved to {OUTPUT_DIR}/xgboost_fraud_model_v2_robust.json")


Model saved to /Users/solikamar/synthea_kenya/output/model/xgboost_fraud_model_v2_robust.json
